### Using Boruta on the Madalon Data Set
Author: [Mike Bernico](mike.bernico@gmail.com)

This example demonstrates using Boruta to find all relevant features in the Madalon dataset, which is an artificial dataset used in NIPS2003 and cited in the [Boruta paper](https://www.jstatsoft.org/article/view/v036i11/v36i11.pdf).

This dataset has 2000 observations and 500 features.  We will use Boruta to identify the features that are relevant to the classification task.





In [1]:
# Installation
!pip install greedyboruta

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from greedyboruta import GreedyBorutaPy

In [4]:
def load_data():
    # URLS for dataset via UCI
    train_data_url='https://archive.ics.uci.edu/ml/machine-learning-databases/madelon/MADELON/madelon_train.data'
    train_label_url='https://archive.ics.uci.edu/ml/machine-learning-databases/madelon/MADELON/madelon_train.labels'

    X_data = pd.read_csv(train_data_url, sep=" ", header=None)
    y_data = pd.read_csv(train_label_url, sep=" ", header=None)
    data = X_data.loc[:, :499]
    data['target'] = y_data[0]
    return data

In [5]:
data = load_data()

In [6]:
data.head()

,0,1,2,3,4,5,6,7,8,9,...,491,492,493,494,495,496,497,498,499,target
0,485,477,537,479,452,471,491,476,475,473,...,481,477,485,511,485,481,479,475,496,-1
1,483,458,460,487,587,475,526,479,485,469,...,478,487,338,513,486,483,492,510,517,-1
2,487,542,499,468,448,471,442,478,480,477,...,481,492,650,506,501,480,489,499,498,-1
3,480,491,510,485,495,472,417,474,502,476,...,480,474,572,454,469,475,482,494,461,1
4,484,502,528,489,466,481,402,478,487,468,...,479,452,435,486,508,481,504,495,511,1


In [7]:
y = data.pop('target')
X = data.copy().values

Boruta conforms to the sklearn api and can be used in a Pipeline as well as on it's own. Here we will demonstrate stand-alone operation.

First we will instantiate an estimator that Boruta will use. Then we will instantiate a Boruta Object.

In [8]:
rf = RandomForestClassifier(n_jobs=-1, class_weight=None, max_depth=7, random_state=0)
# Define Boruta feature selection method
feat_selector = GreedyBorutaPy(rf, n_estimators='auto', verbose=2, random_state=0)

Once built, we can use this object to identify the relevant features in our dataset.

In [9]:
feat_selector.fit(X, y)

Iteration: 	1 / 9
Confirmed: 	19
Tentative: 	481
Rejected: 	0
Iteration: 	2 / 9
Confirmed: 	19
Tentative: 	481
Rejected: 	0
Iteration: 	3 / 9
Confirmed: 	20
Tentative: 	480
Rejected: 	0
Iteration: 	4 / 9
Confirmed: 	20
Tentative: 	480
Rejected: 	0
Iteration: 	5 / 9
Confirmed: 	22
Tentative: 	478
Rejected: 	0
Iteration: 	6 / 9
Confirmed: 	22
Tentative: 	478
Rejected: 	0
Iteration: 	7 / 9
Confirmed: 	22
Tentative: 	478
Rejected: 	0
Iteration: 	8 / 9
Confirmed: 	22
Tentative: 	0
Rejected: 	478


GreedyBorutaPy finished running.

Iteration: 	9 / 9
Confirmed: 	22
Tentative: 	0
Rejected: 	478


GreedyBorutaPy(estimator=RandomForestClassifier(max_depth=7, n_estimators=451,
                                                n_jobs=-1,
                                                random_state=RandomState(MT19937) at 0x22EB8724440),
               n_estimators='auto',
               random_state=RandomState(MT19937) at 0x22EB8724440, verbose=2)

Boruta has confirmed only a few features as useful. When our run ended, Boruta was undecided on 2 features.

We can interrogate `.support_` to understand which features were selected. `.support_` returns an array of booleans that we can use to slice our feature matrix to include only relevant columns. Of course, `.transform` can also be used, as expected in the scikit API.

In [10]:
# Check selected features
print(feat_selector.support_)
# Select the chosen features from our dataframe.
selected = X[:, feat_selector.support_]
print ("")
print ("Selected Feature Matrix Shape")
print (selected.shape)

[False False False False False False False False False False  True False
 False False False False False False False False False False False False
 False False False False  True False False False False False False False
 False False False False False False False False False False False False
  True False False False False False False False False False False False
 False False False False  True False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False  True False False
 False False False False False False False False False False False False
 False False False False False False False False  True False False False
 False False False False False False False False False False False False
 False False False False False False False False False  True False False
 False False False False False False False False Fa

We can also interrogate the ranking of the unselected features with .ranking_

In [11]:
feat_selector.ranking_

array([426, 356, 121, 367,  17, 269, 157, 472, 418, 363,   1, 213, 259,
       104, 302, 336, 285, 174, 110, 413, 345, 263, 436, 281, 299, 134,
       186, 419,   1, 228, 437,  53, 131, 228, 356, 148, 210, 293,  76,
       462, 451,  92, 252,  41,  39, 259, 146, 178,   1, 113, 162, 168,
       397, 423, 195,  27,  24, 195, 384, 158,  62, 315, 375, 452,   1,
       178, 243, 228, 165, 409, 347, 235, 192,  33, 301, 290, 327,  88,
       146, 116, 422, 125, 352, 427, 267,  56, 399, 248, 332,  54, 479,
       242, 365, 279, 203,  85, 447, 268, 110, 168, 204, 382, 421, 266,
       200,   1,  36, 398, 360, 234, 419, 123, 396, 324, 141, 307,  20,
       354, 283,  79, 458, 269, 122, 209, 334, 139, 305,  71,   1, 130,
        70,  63, 313, 445, 153, 151,   9,  12, 423, 103,  94, 298, 324,
       342, 296, 330, 309,  88, 370,  58, 237, 318, 104,   1, 469, 192,
       265, 239, 328, 310, 390,  66, 404, 413,  97, 214, 461, 437, 464,
       355, 224, 119, 124, 454, 368,  40, 365, 205, 245, 316,  5